In [1]:
# -*- coding: utf-8 -*-
from abc import ABC, abstractmethod
from typing import Dict

import torch as th
from torch import nn

from torch import nn
from typing import Callable
import torch as th
from torch.nn.init import xavier_normal_, normal_
from torch.nn import functional as F


In [3]:
def hermite(x: th.Tensor, n: int) -> th.Tensor:
    h_s = [th.ones(*x.size(), device=x.device), x]

    for i in range(1, n):
        h_s.append(x * h_s[i] - i * h_s[i - 1])
    
    return th.einsum(
        "bn...,n->bn...",
        th.slice_copy(th.stack(h_s, dim=1), 1, 1),
        1 / th.exp(th.lgamma(th.arange(2, n + 2, device=x.device)) / 2),
    )



class Hermite(nn.Module):
    def __init__(self, n: int) -> None:
        super().__init__()
        self.__n = n

    def forward(self, x: th.Tensor) -> th.Tensor:
        return hermite(x, self.__n)

    def get_size(self) -> int:
        return self.__n

# Conv

In [7]:
batch_size = 3
in_channels = 2
out_channels = 4
size = 8

kernel_size = 4
stride = 2
padding = 1

n = 5

output_size = (
            size - kernel_size + 2 * padding
        ) // stride + 1

her = Hermite(n)

In [21]:
w_b = th.randn(out_channels, in_channels*kernel_size**2)
w_s = th.randn(out_channels, in_channels*kernel_size**2)
c = th.randn(n, out_channels, in_channels*kernel_size**2)

In [33]:
x_0 = th.randn(batch_size, in_channels, size, size)
print(x_0.size())
out_unfolded = F.unfold(x_0, kernel_size=kernel_size, stride=stride, padding=padding)
print(out_unfolded.size())
out_her = her(out_unfolded)
print(out_her.size())
out_act = th.sum(out_her.unsqueeze(2) * c.unsqueeze(-1), dim=1)
print(out_act.size(), out_unfolded.unsqueeze(1).size(), w_b.unsqueeze(-1).size())
out = th.sum(F.mish(out_unfolded).unsqueeze(1) * w_b.unsqueeze(-1) + w_s.unsqueeze(-1) * out_act, dim=2).unflatten(-1, (output_size, output_size))
print(out.size())

torch.Size([3, 2, 8, 8])
torch.Size([3, 32, 16])
torch.Size([3, 5, 32, 16])
torch.Size([3, 4, 32, 16]) torch.Size([3, 1, 32, 16]) torch.Size([4, 32, 1])
torch.Size([3, 4, 4, 4])


µ# ConvTr

In [108]:
batch_size = 3
in_channels = 2
out_channels = 4
size = 8

kernel_size = 4
stride = 2
padding = 1

n = 5

output_size = stride * (size - 1) + kernel_size - 2 * padding

her = Hermite(n)

In [109]:
output_size

16

In [110]:
x = th.randn(batch_size, in_channels, size, size)

In [117]:
c = th.randn(n, out_channels, in_channels, kernel_size, kernel_size)

In [119]:
o = her(x)
print(o.size(), c.size())
o = th.einsum("baiwh,aoikl->boklwh", o, c).view(batch_size, out_channels * kernel_size**2, size * size)
print(o.size())
o = F.fold(o, output_size, kernel_size=kernel_size, stride=stride, padding=padding)

torch.Size([3, 5, 2, 8, 8]) torch.Size([5, 4, 2, 4, 4])
torch.Size([3, 64, 64])


In [115]:
o.size()

torch.Size([3, 4, 16, 16])

In [ ]:
out = th.sum(x.view(batch_size, in_channels, 1, 1, size * size, 1) * c, dim=-1)
out = th.sum(out, dim=1)
out = out.view(batch_size, out_channels * kernel_size**2, -1)
out = F.fold(out, output_size, kernel_size, dilation=1, padding=padding, stride=stride)

In [ ]:
out.size()